### Import Library & Load Dataset

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, auc

# Load dataset
df = pd.read_csv('course_lead_scoring_2026.csv')
print(f'Dataset shape: {df.shape}')
print(df.head())

Dataset shape: (5000, 9)
      lead_source       industry  ... lead_score converted
0  organic_search     technology  ...       0.64         1
1    social_media     technology  ...       0.72         1
2        referral         retail  ...       0.58         1
3        paid_ads  manufacturing  ...       0.57         0
4        referral  manufacturing  ...       0.62         1

[5 rows x 9 columns]


### Data Preparation

In [2]:
# Check missing values
print("Missing values before imputation:")
print(df.isnull().sum())

# Define categorical and numerical features
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

# Impute missing values: 'NA' for categorical, 0.0 for numerical
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

print("\nMissing values after imputation:")
print(df.isnull().sum())

# Split dataset into full_train, val, test (60% train, 20% val, 20% test)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

print(f'\nTrain shape: {len(df_train)}, Validation shape: {len(df_val)}, Test shape: {len(df_test)}')

Missing values before imputation:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

Missing values after imputation:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

Train shape: 3000, Validation shape: 1000, Test shape: 1000


### Q1. ROC AUC Feature Importance

In [3]:
# Q1: Compute ROC AUC for each numerical variable on the training dataset
auc_scores = {}

for col in numerical:
    score = roc_auc_score(y_train, df_train[col])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[col])
    auc_scores[col] = score
    print(f'{col:25} -> AUC: {score:.4f}')

best_feature = max(auc_scores, key=auc_scores.get)
print(f'\nNumerical variable with the highest AUC: {best_feature} ({auc_scores[best_feature]:.4f})')

annual_income             -> AUC: 0.6082
number_of_courses_viewed  -> AUC: 0.7230
interaction_count         -> AUC: 0.7649
lead_score                -> AUC: 0.7882

Numerical variable with the highest AUC: lead_score (0.7882)


### Q2. Training the Model

In [4]:
# Q2: Apply one-hot encoding using DictVectorizer and train Logistic Regression
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, y_pred)

print(f'Validation AUC: {val_auc:.4f}')
print(f'Validation AUC (rounded to 3 digits): {val_auc:.3f}')

Validation AUC: 0.7318
Validation AUC (rounded to 3 digits): 0.732


### Q3. Precision and Recall

In [5]:
# Q3: Evaluate model on thresholds from 0.0 to 1.0 with step 0.01
thresholds = np.arange(0.0, 1.01, 0.01)

results = []

actual_pos = (y_val == 1)
actual_neg = (y_val == 0)

for t in thresholds:
    t_round = round(t, 2)
    predict_pos = (y_pred >= t_round)
    predict_neg = (y_pred < t_round)
    
    tp = (predict_pos & actual_pos).sum()
    fp = (predict_pos & actual_neg).sum()
    fn = (predict_neg & actual_pos).sum()
    tn = (predict_neg & actual_neg).sum()
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    diff = abs(p - r)
    
    results.append({'threshold': t_round, 'precision': p, 'recall': r, 'difference': diff})

df_pr = pd.DataFrame(results)

# Find intersection (smallest absolute difference where metrics are non-zero)
valid_pr = df_pr[(df_pr['precision'] > 0) | (df_pr['recall'] > 0)]
intersect_row = valid_pr.loc[valid_pr['difference'].idxmin()]

print(f"Intersection threshold: {intersect_row['threshold']:.2f}")
print(f"Precision: {intersect_row['precision']:.4f}, Recall: {intersect_row['recall']:.4f}, Difference: {intersect_row['difference']:.4f}\n")

for t_opt in [0.43, 0.63, 0.73, 0.83]:
    row = df_pr[df_pr['threshold'] == t_opt].iloc[0]
    print(f"Threshold {t_opt:.2f} -> Precision: {row['precision']:.4f}, Recall: {row['recall']:.4f}, Difference: {row['difference']:.4f}")

Intersection threshold: 0.63
Precision: 0.7328, Recall: 0.7253, Difference: 0.0075

Threshold 0.43 -> Precision: 0.6225, Recall: 0.9625, Difference: 0.3399
Threshold 0.63 -> Precision: 0.7328, Recall: 0.7253, Difference: 0.0075
Threshold 0.73 -> Precision: 0.8233, Recall: 0.4454, Difference: 0.3780
Threshold 0.83 -> Precision: 0.8796, Recall: 0.1621, Difference: 0.7175


### Q4. F1 Score

In [6]:
# Q4: Compute F1 score for all thresholds
df_pr['f1'] = np.where(
    (df_pr['precision'] + df_pr['recall']) > 0,
    2 * (df_pr['precision'] * df_pr['recall']) / (df_pr['precision'] + df_pr['recall']),
    0
)

max_f1_row = df_pr.loc[df_pr['f1'].idxmax()]
print(f"Maximal F1 threshold: {max_f1_row['threshold']:.2f} with F1 score = {max_f1_row['f1']:.4f}\n")

for t_opt in [0.21, 0.41, 0.61, 0.81]:
    row = df_pr[df_pr['threshold'] == t_opt].iloc[0]
    print(f"Threshold {t_opt:.2f} -> F1 Score: {row['f1']:.4f}")

Maximal F1 threshold: 0.41 with F1 score = 0.7576

Threshold 0.21 -> F1 Score: 0.7390
Threshold 0.41 -> F1 Score: 0.7576
Threshold 0.61 -> F1 Score: 0.7451
Threshold 0.81 -> F1 Score: 0.3397


### Q5. 5-Fold Cross-Validation

In [7]:
# Q5: 5-Fold Cross-Validation with KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(df_full_train), 1):
    df_train_fold = df_full_train.iloc[train_idx]
    df_val_fold = df_full_train.iloc[val_idx]
    
    y_train_fold = df_train_fold.converted.values
    y_val_fold = df_val_fold.converted.values
    
    dv_fold = DictVectorizer(sparse=False)
    X_train_fold = dv_fold.fit_transform(df_train_fold[categorical + numerical].to_dict(orient='records'))
    X_val_fold = dv_fold.transform(df_val_fold[categorical + numerical].to_dict(orient='records'))
    
    model_fold = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    model_fold.fit(X_train_fold, y_train_fold)
    
    y_pred_fold = model_fold.predict_proba(X_val_fold)[:, 1]
    auc_fold = roc_auc_score(y_val_fold, y_pred_fold)
    scores.append(auc_fold)
    print(f'Fold {fold}: AUC = {auc_fold:.4f}')

mean_score = np.mean(scores)
std_score = np.std(scores)

print(f'\nMean AUC: {mean_score:.4f}')
print(f'Standard deviation of AUC scores: {std_score:.4f}')
print(f'Standard deviation (rounded to 3 digits): {round(std_score, 3)}')

Fold 1: AUC = 0.7306
Fold 2: AUC = 0.7296
Fold 3: AUC = 0.7363
Fold 4: AUC = 0.7412
Fold 5: AUC = 0.7209

Mean AUC: 0.7317
Standard deviation of AUC scores: 0.0069
Standard deviation (rounded to 3 digits): 0.007


### Q6. Hyperparameter Tuning

In [8]:
# Q6: Hyperparameter tuning for C in [0.000001, 0.001, 1] using 5-Fold CV
C_values = [0.000001, 0.001, 1]
results_c = {}

for C in C_values:
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    fold_scores = []
    
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_fold = df_full_train.iloc[train_idx]
        df_val_fold = df_full_train.iloc[val_idx]
        
        y_train_fold = df_train_fold.converted.values
        y_val_fold = df_val_fold.converted.values
        
        dv_fold = DictVectorizer(sparse=False)
        X_train_fold = dv_fold.fit_transform(df_train_fold[categorical + numerical].to_dict(orient='records'))
        X_val_fold = dv_fold.transform(df_val_fold[categorical + numerical].to_dict(orient='records'))
        
        model_fold = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
        model_fold.fit(X_train_fold, y_train_fold)
        
        y_pred_fold = model_fold.predict_proba(X_val_fold)[:, 1]
        auc_fold = roc_auc_score(y_val_fold, y_pred_fold)
        fold_scores.append(auc_fold)
        
    mean_c = np.mean(fold_scores)
    std_c = np.std(fold_scores)
    results_c[C] = (mean_c, std_c)
    print(f'C = {C:<8} -> Mean AUC: {mean_c:.4f} (round: {round(mean_c, 3)}), Std: {std_c:.4f} (round: {round(std_c, 3)})')

best_C = max(results_c, key=lambda c: (round(results_c[c][0], 3), -round(results_c[c][1], 3)))
print(f'\nBest parameter C: {best_C} with Mean AUC: {round(results_c[best_C][0], 3)} and Std: {round(results_c[best_C][1], 3)}')

C = 1e-06    -> Mean AUC: 0.6167 (round: 0.617), Std: 0.0187 (round: 0.019)
C = 0.001    -> Mean AUC: 0.7492 (round: 0.749), Std: 0.0100 (round: 0.01)
C = 1        -> Mean AUC: 0.7317 (round: 0.732), Std: 0.0069 (round: 0.007)

Best parameter C: 0.001 with Mean AUC: 0.749 and Std: 0.01
